In [ ]:
import os, copy
import glob

import numpy as np
import scipy.optimize as so
import pandas as pd
import xarray as xr
import rioxarray as rxr
import datetime as dt
import cartopy.crs as ccrs

import netCDF4
import h5py
from osgeo import gdal

#%matplotlib widget
import matplotlib as mpl
import matplotlib.pyplot as plt
import colorcet as cc


import hgrs
import grstbx

opj = os.path.join
hgrs.__version__

### Set PRISMA file paths
Give path for L1C (L2C is normally in the same directory). The L2C is needed to get the viewing geometry parameters (viewing zenith angle, sun zenith angle and relative azimuth)


In [ ]:
l1c_path ='/data/satellite/prisma/albufera/L1/PRS_L1_STD_OFFL_20240723110233_20240723110237_0001.he5' # 31122105559_20231122105603_0001.he5'
l1c_path ='/data/satellite/prisma/L1/PRS_L1_STD_OFFL_20250124154311_20250124154315_0001.he5'
l1c_path ='/data/satellite/prisma/garaba/PRS_L1_STD_OFFL_20230226103653_20230226103658_0001.he5'


l2c_path = l1c_path.replace('L1_STD_OFFL', 'L2C_STD')
odir='/data/satellite/prisma/L2A'
lev="L2A"
basename = os.path.basename(l1c_path)

outfile = basename.replace('L1_STD_OFFL', lev)
outfile = outfile.replace('.he5', '.nc').rstrip('/')
outfile = os.path.join(odir, outfile)

driver = hgrs.Driver('prisma')

l1c_prod = driver.driver(l1c_path,l2c_path,reflectance_unit=True,geoproject=True,parallel=False)
l1c_prod.Rtoa.load()
l1c_prod

In [ ]:
raster = l1c_prod.sza.rio.reproject(4326)
clon, clat = float(raster.x.mean()),float(raster.y.mean())
print(clon, clat)

In [ ]:

fig = l1c_prod.Rtoa.sel(wl=[440,550,660,770,880,1600], method='nearest').rio.reproject(4326).plot.imshow(
    col='wl', col_wrap=3, cmap=plt.cm.Spectral_r, robust=True
)
#plt.savefig(root_path+'figure_rtoa.jpg', format='jpg', dpi=300)
plt.show()

In [ ]:
str_epsg = str(l1c_prod.rio.crs)
crs = l1c_prod.rio.crs
zone = str_epsg[-2:]
is_south = str_epsg[2] == 7
proj = ccrs.UTM(zone, is_south)

coarsening=1
gamma=0.2
brightness_factor = 1
plt.figure(figsize=(15,15))
fig = (l1c_prod.Rtoa[:, ::coarsening, ::coarsening].sel(wl=[705,560,460],method='nearest')**gamma*brightness_factor).plot.imshow(rgb='wl',robust=True, subplot_kws=dict(projection= proj))
fig.axes.set(xticks=[], yticks=[])
fig.axes.set_ylabel('')
fig.axes.set_xlabel('')
fig

In [ ]:
import logging
cams_dir = '/data/cams/world/'
cams_path = opj(cams_dir,'cams_forecast_2024-07.nc')
lut_file='/DATA/git/satellite_app/hgrs/data/lut/opac_osoaa_lut_v3_light.nc'
aero_lut = xr.open_dataset(lut_file)
aero_lut['wl'] = aero_lut['wl'] * 1000

date = l1c_prod.time

# -----------------------------------------
# Create hGRS object
# -----------------------------------------
logging.info('Create hGRS object')
prod = hgrs.Algo(l1c_prod, xcoarsen=20, ycoarsen=20, expon=1)
prod.round_angles()

# -----------------------------------------
# get CAMS and set atmospheric parameters
# -----------------------------------------
logging.info('get CAMS and set atmospheric parameters')
# lazy loading
cams = xr.open_dataset(cams_path, decode_cf=True,
                       chunks={'time': 1, 'x': 500, 'y': 500})
# slicing
#
cams = cams.sel(latitude=clat, longitude=clon, method='nearest')
# fix for new ADS format (sept 2024)
if ('forecast_period' in cams.dims) & ('forecast_reference_time' in cams.dims):
    cams = cams.stack(time_buffer=['forecast_period', 'forecast_reference_time']).swap_dims(
        {'time_buffer': 'valid_time'}).sortby('valid_time').rename(
        {'valid_time': 'time'}).drop_vars(['time_buffer'])

cams = cams.sel(time=date, method='nearest')

# select OPAC aerosol model
# aod = cams[['aod355', 'aod380', 'aod400', 'aod440', 'aod469', 'aod500', 'aod550', 'aod645', 'aod670',
#            'aod800', 'aod865', 'aod1020', 'aod1064', 'aod1240', 'aod1640', 'aod2130']].to_pandas()
# aod.index = aod.index.str.replace('aod', '').astype(int)
# cams_aod = aod.to_xarray().rename({'index': 'wl'})
cams_wls = [469, 550, 670, 865, 1240]
param_aod = []
for wl in cams_wls:
    wl_ = str(wl)
    param_aod.append('aod' + wl_)

cams_aod = cams[param_aod].to_array(dim='wl')

wl_cams = cams_aod.wl.str.replace('aod', '').astype(float)
cams_aod = cams_aod.assign_coords(wl=wl_cams)

In [ ]:
rh = '_rh70'
models = ['COAV' + rh, 'COPO' + rh, 'DESE' + rh, 'MACL' + rh, 'MAPO' + rh,
          ]  # 'ANTA' + rh, 'ARCT' + rh,'URBA' + rh
lut_aod = aero_lut.aot.sel(model=models, aot_ref=1).interp(wl=cams_aod.wl)
idx = np.abs((cams_aod / cams.aod550) - lut_aod).sum('wl').argmin()
opac_model = aero_lut.sel(model=models).model.values[idx]

# set gases and pressure
prod.pressure = float(cams.sp) * 1e-2
prod.to3c = float(cams.gtco3)
prod.tno2c = float(cams.tcno2)
prod.tch4c = float(cams.tc_ch4)

In [ ]:
# -----------------------------------------
# Apply water masking
# -----------------------------------------
logging.info('Apply water masking')
prod.apply_water_masks()

# -----------------------------------------
# Construct coarse resolution raster
# -----------------------------------------
logging.info('Construct coarse resolution raster')
prod.get_coarse_masked_raster()
# prod.plot_water_pix_number()

# -----------------------------------------
# Correct for gaseous absorption
# -----------------------------------------
logging.info('Correct for gaseous absorption')
prod.get_gaseous_transmittance()
prod.other_gas_correction()

In [ ]:
Tg = prod.spectral.convolve2(np.exp(- prod.air_mass_mean * prod.abs_gas_opt_thick),name='Tg')

In [ ]:
prod.Tg_other.plot(marker='o',ms=2.5,lw=0.5)
Tg.plot(marker='o',ms=2.5,lw=0.5)

In [ ]:
plt.figure(figsize=(15,15))
prod.raster.Rtoa[:, ::coarsening, ::coarsening].sel(wl=[705,560,460],method='nearest').plot.imshow(rgb='wl',robust=True, subplot_kws=dict(projection= proj))

In [ ]:
logging.info('water vapor retrieval and correction')
wv_retrieval = hgrs.WaterVapor(prod)
wv_retrieval.solve()

In [ ]:
wv_retrieval.water_vapor.tcwv.plot.imshow()

In [ ]:
prod.get_wv_transmittance_raster(wv_retrieval.water_vapor)
prod.water_vapor_correction()

In [ ]:
prod.Twv_raster.mean(['x','y']).plot(marker='o',ms=2.5,lw=0.5)
prod.Tg_other.plot(marker='o',ms=2.5,lw=0.5)

In [ ]:
# ------------------------------------------
# aerosol retrieval
# ------------------------------------------
logging.info('aerosol retrieval')

variable = 'Rtoa'

prod.coarse_masked_raster = prod.remove_wl_dataset(
    prod.coarse_masked_raster, prod.wl_to_remove, variable=variable)

# TODO double check regularization from CAMS AOT values
aod550_mean = cams.aod550.mean().values

aod550_std = cams.aod550.std().values
aod550_std = np.max([aod550_std, 0.2 * aod550_mean+0.05])
aot550_min = np.max([aod550_mean - 2*aod550_std,0.0017])
aero_retrieval = hgrs.Aerosol(prod,
                              aerosol_model=opac_model,
                              first_guess=[aod550_mean, 0.],
                              aot550_limits=[aot550_min,
                                             aod550_mean + aod550_std])
aero_retrieval.solve()
aero_retrieval.get_atmo_parameters(prod.coarse_masked_raster.wl)

In [ ]:
aero_retrieval.atmo_img.Rtoa_diff.sel(wl=550,method='nearest').plot.imshow(robust=True, subplot_kws=dict(projection= proj))

In [ ]:
# ------------------------------------------
# full resolution processing
# ------------------------------------------
logging.info('process full resolution')

prod.raster = prod.remove_wl_dataset(prod.raster, prod.wl_to_remove)
prod.other_gas_correction(raster_name='raster', variable='Rtoa')

In [ ]:
prod.raster['Rtoa']#/prod.Twv_raster#.interp(wl=prod.raster.wl)#=prod.raster['Ltoa']/prod.raster['F0']

In [ ]:
plt.figure(figsize=(15,15))
prod.raster.Rtoa[:, ::coarsening, ::coarsening].sel(wl=[705,560,460],method='nearest').plot.imshow(rgb='wl',robust=True, subplot_kws=dict(projection= proj))

In [ ]:
wv_full = prod.get_full_resolution(wv_retrieval.water_vapor)
prod.get_wv_transmittance_raster(wv_full)

In [ ]:
prod.water_vapor_correction(raster_name='raster', variable='Rtoa')

In [ ]:
wv_full.tcwv#.plot.imshow()

In [ ]:
Rdiff_full

In [ ]:
Rdiff_full = aero_retrieval.atmo_img.Rtoa_diff.interp(x=prod.raster.x, y=prod.raster.y)
Tdir_full = aero_retrieval.atmo_img.Tdir.interp(x=prod.raster.x, y=prod.raster.y)
Rcorr = (prod.raster.Rtoa - Rdiff_full)
wl_sunglint = prod.wl_sunglint
sunglint_eps = aero_retrieval.sunglint_eps
BRDF_sunglint = (Rcorr.sel(wl=wl_sunglint) / (Tdir_full.sel(wl=wl_sunglint)
                                              * sunglint_eps.sel(wl=wl_sunglint))).mean(dim='wl')
BRDF_sunglint.name = 'brdfg_full'
BRDF_sunglint = BRDF_sunglint.reset_coords().brdfg_full
# TODO clean up xarray inheritance of some extra coordinates...
# BRDF_sunglint = BRDF_sunglint.drop_vars('aot_ref', errors=False).squeeze()
Rdir = Tdir_full * sunglint_eps * BRDF_sunglint

Rrs_l2 = (Rcorr - Rdir) / np.pi

In [ ]:
Rdiff_full.sel(wl=550,method='nearest').plot.imshow(robust=True, subplot_kws=dict(projection= proj))

In [ ]:
(prod.raster.Rtoa )[:, ::coarsening, ::coarsening].sel(wl=[705,560,460],method='nearest').plot.imshow(rgb='wl',robust=True, subplot_kws=dict(projection= proj))

In [ ]:
Rrs_l2

In [ ]:
# finally correct for down and upward transmittances
# TODO compute pixel wise
Ttot_Ed = xr.open_dataset('/DATA/git/satellite_app/hgrs/data/lut/transmittance_downward_irradiance.nc')
aot_ref = float(aero_retrieval.aero_img.aot_ref.mean())
wl = Rrs_l2.wl.values
sza = float(Rrs_l2.sza)
vza = float(Rrs_l2.vza)
Ttot_Ed_ = Ttot_Ed.Ttot_Ed.sel(model=opac_model).interp(sza=sza, method='cubic'
                                                        ).interp(aot_ref=aot_ref, method='quadratic').interp(
    wl=wl, method='cubic')
Ttot_Lu_ = Ttot_Ed.Ttot_Ed.sel(model=opac_model).interp(sza=vza, method='cubic'
                                                        ).interp(aot_ref=aot_ref, method='quadratic').interp(
    wl=wl, method='cubic') ** 1.05
Ttot = (Ttot_Ed_ * Ttot_Lu_).reset_coords(drop=True)
Rrs_l2 = Rrs_l2 / Ttot
Rrs_l2.name = 'Rrs'

In [ ]:
# -----------------------------
# construct output image
# -----------------------------
logging.info('construct final product')

# -----------------------------
# data
wv = wv_retrieval.water_vapor.rename({"x": "xc", "y": "yc"})
aero = aero_retrieval.aero_img.rename({"x": "xc", "y": "yc"})
water_pixel_prop = (prod.coarse_masked_raster.water_pixel_number / prod.Npix_per_megapix).drop_vars(
    'tcwv').rename({"x": "xc", "y": "yc"})
water_pixel_prop.name = 'water_pix_prop'
#geom = prod.raster[['lon', 'lat']].drop_vars('tcwv')
Rrs_ = Rrs_l2.reset_coords().drop_vars(['model', 'z']).rename({'tcwv': 'tcwv_full', 'aot_ref': 'aot_ref_full'}).set_coords(['time','spatial_ref'])
l2_prod = xr.merge([Rrs_, wv, aero,water_pixel_prop])
l2_prod['brdfg_full'] = BRDF_sunglint



param = 'Rrs'
l2_prod[param].attrs['unit'] = 'per steradian'
l2_prod[param].attrs['long_name'] = 'Remote sensing reflectance'
l2_prod[param].attrs['description'] = 'Directional water-leaving radiance normalized ' + \
                                      'by downwelling irradiance in the observation geometry'

param = 'water_pix_prop'
l2_prod[param].attrs['unit'] = '-'
l2_prod[param].attrs['description'] = 'Relative number of water pixel within mega-pixel used for inversion'

param = 'brdfg'
l2_prod[param].attrs['unit'] = '-'
l2_prod[param].attrs['long_name'] = 'BRDF_sunglint'
l2_prod[param].attrs['description'] = 'Bidirectional reflectance distribution function ' + \
                                      'estimated from the sunglint in the SWIR for the observation geometry'
param = 'brdfg_std'
l2_prod[param].attrs['unit'] = '-'
l2_prod[param].attrs['long_name'] = 'BRDF_sunglint_standard deviation'
l2_prod[param].attrs['description'] = 'Uncertainty based on optimal estimation procedure'
param = 'brdfg_full'
l2_prod[param].attrs['unit'] = '-'
l2_prod[param].attrs['long_name'] = 'BRDF_sunglint'
l2_prod[param].attrs['description'] = 'Bidirectional reflectance distribution function ' + \
                                      'estimated from the sunglint in the SWIR for the observation geometry'

param = 'aot_ref'
l2_prod[param].attrs['unit'] = '-'
l2_prod[param].attrs['long_name'] = 'aerosol_optical_thickness'
l2_prod[param].attrs['description'] = 'Aerosol optical thickness at the reference wavelength (550nm)'
param = 'aot_ref_std'
l2_prod[param].attrs['unit'] = '-'
l2_prod[param].attrs['long_name'] = 'aerosol_optical_thickness_standard_deviation'
l2_prod[param].attrs['description'] = 'Uncertainty based on optimal estimation procedure'
param = 'aot_ref_full'
l2_prod[param].attrs['unit'] = '-'
l2_prod[param].attrs['long_name'] = 'aerosol_optical_thickness'
l2_prod[param].attrs['description'] = 'Aerosol optical thickness at the reference wavelength (550nm)'

param = 'tcwv'
l2_prod[param].attrs['unit'] = 'kg m-2'
l2_prod[param].attrs['long_name'] = 'total_columnar_water_vapor'
l2_prod[param].attrs['description'] = 'Water vapor integrated over the atmospheric layer'
param = 'tcwv_std'
l2_prod[param].attrs['unit'] = 'kg m-2'
l2_prod[param].attrs['long_name'] = 'total_columnar_water_vapor_standard_deviation'
l2_prod[param].attrs['description'] = 'Uncertainty based on optimal estimation procedure'
param = 'tcwv_full'
l2_prod[param].attrs['unit'] = 'kg m-2'
l2_prod[param].attrs['long_name'] = 'total_columnar_water_vapor'
l2_prod[param].attrs['description'] = 'Water vapor integrated over the atmospheric layer'

l2_prod['pressure'] = prod.pressure
l2_prod['pressure'].attrs['unit'] = 'hPa'
l2_prod['pressure'].attrs['description'] = 'Atmospheric pressure at the surface level'
l2_prod['pressure'].attrs['source'] = 'computed from CAMS and DEM (see DEM metadata)'

param = 'to3c'
l2_prod[param] = prod.__dict__[param]
l2_prod[param].attrs['unit'] = ''
l2_prod[param].attrs['description'] = 'Total columnar ozone concentration'
l2_prod[param].attrs['source'] = 'CAMS'

param = 'tno2c'
l2_prod[param] = prod.__dict__[param]
l2_prod[param].attrs['unit'] = ''
l2_prod[param].attrs['description'] = 'Total columnar Nitrogen dioxide concentration'
l2_prod[param].attrs['source'] = 'CAMS'

# -----------------------------
# --metadata
l2_prod.attrs = prod.raster.attrs
l2_prod.attrs['processing_date'] = str(dt.datetime.now())
l2_prod.attrs['acquisition_date'] =str(l1c_prod.time )#attrs['acquisition_date'])
l2_prod.attrs['hgrs_version'] = hgrs.__version__
l2_prod.attrs['description'] = 'PRISMA L2A-hGRS cube data'
l2_prod.attrs['DEM'] = 'not available'
l2_prod.attrs['aerosol_model'] = aero_retrieval.aerosol_model
keys = ['wl_water_vapor', 'wl_sunglint', 'wl_atmo', 'wl_to_remove', 'wl_green', 'wl_nir', 'wl_1600', 'wl_rgb',
        'xcoarsen', 'ycoarsen', 'Npix_per_megapix', 'block_size', 'pixel_percentage', 'pixel_threshold',
        'ang_resol', 'dirdata', 'abs_gas_file', 'lut_file', 'water_vapor_transmittance_file',
        'sunglint_threshold',
        'ndwi_threshold', 'green_swir_index_threshold', 'pressure', 'to3c', 'tno2c', 'tch4c', 'psl',
        'coef_abs_scat',
        'altitude']
for key in keys:
    l2_prod.attrs[key] = str(prod.__dict__[key])



In [ ]:
l2_prod

In [ ]:
def compute_scale_and_offset(array, nbit=16):
    max_, min_=np.nanmax(array),np.nanmin(array)
    # stretch/compress data to the available packed range
    scale_factor = (max_ - min_) / (2 ** nbit - 1)
    # translate the range to be symmetric about zero
    add_offset = min_ + 2 ** (nbit - 1) * scale_factor
    return scale_factor, add_offset

scale_factor, add_offset = compute_scale_and_offset(l2_prod.Rrs)

In [ ]:
complevel = 5
encoding = {
            'Rrs': {'dtype': 'int16', 'scale_factor': scale_factor, 'add_offset': add_offset, '_FillValue': -32768, "zlib": True,
                    "complevel": complevel},
            'aot_ref_full': {'dtype': 'int16', 'scale_factor': 0.001, '_FillValue': -9999, "zlib": True,
                             "complevel": complevel},
            'aot_ref': {'dtype': 'int16', 'scale_factor': 0.001, '_FillValue': -9999, "zlib": True,
                        "complevel": complevel},
            'aot_ref_std': {'dtype': 'int16', 'scale_factor': 0.001, '_FillValue': -9999, "zlib": True,
                            "complevel": complevel},
            'brdfg_full': {'dtype': 'int16', 'scale_factor': 0.00001, 'add_offset': .2, '_FillValue': -32768,
                           "zlib": True, "complevel": complevel},
            'brdfg': {'dtype': 'int16', 'scale_factor': 0.00001, 'add_offset': .2, '_FillValue': -32768, "zlib": True,
                      "complevel": complevel},
            'brdfg_std': {'dtype': 'int16', 'scale_factor': 0.00001, 'add_offset': .2, '_FillValue': -32768,
                          "zlib": True, "complevel": complevel},
            'tcwv_full': {'dtype': 'int16', 'scale_factor': 0.01, '_FillValue': -9999, "zlib": True,
                          "complevel": complevel},
            'tcwv': {'dtype': 'int16', 'scale_factor': 0.01, '_FillValue': -9999, "zlib": True, "complevel": complevel},
            'tcwv_std': {'dtype': 'int16', 'scale_factor': 0.01, '_FillValue': -9999, "zlib": True,
                         "complevel": complevel}}

In [ ]:


l2_prod.to_netcdf(outfile,encoding=encoding) # 

In [ ]:
xr.open_dataset(outfile, decode_coords='all')

In [ ]:
l2_prod.aot_ref.plot.imshow(cmap=plt.cm.Spectral_r,vmin=0, robust=True)

In [ ]:
l2_prod.Rrs.sel(wl=[440,550,660,770,880,1600], method='nearest').plot.imshow(
    col='wl', col_wrap=3, cmap=plt.cm.Spectral_r,vmin=0, robust=True
)

In [ ]:
import panel as pn

#hv.extension('bokeh')
pn.extension()
from grstbx import visual

v=visual.ViewSpectral(l2_prod.Rrs.isel(wl=range(0,93,5)) ,reproject=True)
print(v)
v.minmax=[0,0.1]
v.minmaxvalues=(0,0.04)
v.visu()

In [ ]:
Twv =prod.Twv_raster.mean(['x','y']) #.plot(marker='o',ms=2.5,lw=0.5)

In [ ]:
import geopandas as gpd

poi_stream = v.poi_stream
geom = poi_stream.data
geom=gpd.points_from_xy(geom['x'],geom['y'])#,crs="EPSG3857")index=[0], c
geom =gpd.GeoDataFrame(crs=3857, geometry=geom).to_crs(crs)
Ndata = len(geom)
cmap = mpl.colors.LinearSegmentedColormap.from_list("",
                                                    ['navy', "blue", 'lightskyblue',
                                                     "grey",   'forestgreen','yellowgreen',
                                                     "khaki", "gold",
                                                     'orangered', "firebrick", 'purple'])
norm = mpl.colors.Normalize(vmin=0.0,vmax=Ndata-1)


wl_range=slice(400,900)
alpha=0.5
lw=1.5
ms=3.5

fig,axs = plt.subplots(1,2,figsize=(15,5))#,sharey=True)   
fig.subplots_adjust(hspace=0.05, wspace=0.25)
ax=axs[0]
indexes=np.dstack([geom.geometry.x,geom.geometry.y])[0]
for ii,index in enumerate(indexes):
    color=cmap(norm(ii))
   
    
    boa=l2_prod.Rrs.sel(x=index[0],y=index[1],method='nearest')
    boa.plot(label='pnt-'+str(ii),marker='o',ms=ms,lw=lw,c=color,alpha=alpha,ax=ax)
    boa.sel(wl=wl_range).plot(label='pnt-'+str(ii),marker='o',ms=ms,lw=lw,alpha=alpha,c=color,ax=axs[1])
#(0.04*prod.Tg_other).plot(marker='o',ms=2.5,lw=0.5,ax=axs[0])
#(0.04*prod.Tg_other).sel(wl=wl_range).plot(marker='o',ms=2.5,lw=0.5,ax=axs[1])
#(0.04*Twv).plot(marker='o',ms=2.5,lw=0.5,ax=axs[0])
#(0.04*Twv).sel(wl=wl_range).plot(marker='o',ms=2.5,lw=0.5,ax=axs[1])

for ax in  axs:
    ax.set_ylabel(r'$R_{rs}\ (sr^{-1})\ or\ R_{TOA}\ (-)$')
    ax.axhline(0,color='k')
    ax.set_xlabel(r'$Wavelength\ (nm)$')
    ax.set_title('')
    ax.minorticks_on() 
    #prod.Twv_raster.mean(['x','y']).plot(marker='o',ms=2.5,lw=0.5)
   

axs[0].legend(fontsize=12)

In [ ]:
raster = l2_prod
Npoly = len(v.aoi_stream.data['xs'])
for index in range(Npoly):
    geom_ = v.get_geom(v.aoi_stream,crs=raster.rio.crs,index=index)
    
    raster_clipped = xr.Dataset()
    
    for param in ['Rrs']:# raster.keys():
        da = raster[param]
        if 'x' in da.dims and 'y' in da.dims:
            raster_clipped[param]=da.rio.clip(geom_.geometry.values)
        else:
            raster_clipped[param]=da
    raster_clipped.attrs = raster.attrs
    
    
    stacked = raster_clipped.Rrs.sel(wl=slice(400,31000)).stack(gridcell=["y", "x"]).dropna('gridcell',thresh=0)
    
    
    group_coord ='wl'
    stat_coord='gridcell'
    stats = xr.Dataset({'median':stacked.groupby(group_coord).median(stat_coord)})
    stats['q25'] = stacked.groupby(group_coord).quantile(0.05,dim=stat_coord)
    stats['q75'] = stacked.groupby(group_coord).quantile(0.95,dim=stat_coord)
    stats['min'] = stacked.groupby(group_coord).min(stat_coord)
    stats['max'] = stacked.groupby(group_coord).max(stat_coord)
    stats['mean'] = stacked.groupby(group_coord).mean(stat_coord)
    stats['std'] = stacked.groupby(group_coord).std(stat_coord)
    stats['pix_num'] = stacked.count(stat_coord)
    
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    
    bands=[3,2,1]
    fig, axs = plt.subplots(1,2, figsize=(18, 6))#,sharey=True
    
    
    ax=axs[0]
    date = stats.time.dt.date.values
    axins = inset_axes(ax, width="45%", height="60%",
                   bbox_to_anchor=(.55, .4, 0.9, 0.9),
                   bbox_transform=ax.transAxes, loc=3)
    
    
    raster_clipped.Rrs.isel(wl=bands).plot.imshow(robust=True,ax=axins)
    axins.set_title('')
    axins.set_axis_off()
    for ii, ax in enumerate(axs):
        stats_=stats
        if ii == 1:
            stats_ =stats.sel(wl=slice(400,900))
        
        ax.minorticks_on()
        ax.axhline(y=0,color='k',lw=1)
        ax.plot(stats_.wl,stats_['median'],c='k',marker='o',ms=4)
        ax.plot(stats_.wl,stats_['mean'],c='red',ls='--',marker='o',ms=4)
        ax.fill_between(stats_.wl, stats_['q25'], stats_['q75'],alpha=0.3,color='grey')
        
        ax.set_xlabel(r'$Wavelength\ (nm)$')
        ax.set_ylabel(r'$R_{rs}\ (sr^{-1})$')
        ax.set_title(date)
    axs[1].set_xlim(400,900)
    plt.show()

In [ ]:
plt.figure(figsize=(15,15))
(l2_prod.Rrs.sel(wl=[705,560,460],method='nearest')).plot.imshow(rgb='wl',robust=True, subplot_kws=dict(projection= proj))

In [ ]:
plt.figure(figsize=(15,15))
l2_prod.brdfg_full.plot.imshow(robust=True, subplot_kws=dict(projection= proj),cbar_kwargs={'shrink': 0.64},cmap=plt.cm.YlGnBu_r)

In [ ]:
import GRSl2bgen
owt_process = GRSl2bgen.OWT_process(l2_prod)

owt_process.execute()

In [ ]:
owt_process.output.owt_dist_Bi2024


In [ ]:


params = ['owt_dist_Spyrakos2018','owt_dist_Bi2024','owt_index_Spyrakos2018','owt_index_Bi2024']
raster = owt_process.output
fig = plt.figure(figsize=(25, 18))

for ii, param in enumerate(params):
    ax = plt.subplot(2,2, ii+1, projection=proj)
    raster[param].plot.imshow(robust=True,cmap=plt.cm.Spectral_r,cbar_kwargs={'shrink': 0.64},ax=ax)

In [ ]:

fig = Rrs_.Rrs.sel(wl=[440,550,660,770,880,1600], method='nearest').rio.reproject(4326).plot.imshow(
    col='wl', col_wrap=3, cmap=plt.cm.Spectral_r, robust=True
)
#plt.savefig(root_path+'figure_rtoa.jpg', format='jpg', dpi=300)
plt.show()

### Set output directory
The output L2A image will be saved in hte directory¶

In [ ]:
odir='/data/satellite/prisma/zoffoli/L2A'
outfile = os.path.join(odir,l2a_name)

### Set CAMS file
CAMS file should cover the date of the image acquisition

In [ ]:
cams_file = "/data/cams/world/cams_forecast_2021-12.nc"

In [ ]:
process_ = hgrs.Process()
process_.execute(l1c_path,
                 l2c_path,
                 cams_file,
                 
                 )


Check the output L2A product (xarray.Dataset)

In [ ]:
process_.l2_prod

If you want to save the output L2A image into netcdf:

In [ ]:
process_.write_output(outfile)

### check L1C data

In [ ]:
coarsening=1
gamma=0.2
brightness_factor = 1
fig = (l1c_prod.Rtoa[:, ::coarsening, ::coarsening].isel(wl=[30,20,10])**gamma*brightness_factor).plot.imshow(rgb='wl',robust=True)#, subplot_kws=dict(projection= l1c.proj))
fig.axes.set(xticks=[], yticks=[])
fig.axes.set_ylabel('')
fig.axes.set_xlabel('')
fig

In [ ]:
### check L2A output

In [ ]:
#(Rcorr[:, ::coarsening, ::coarsening].isel(wl=[30,20,10])*brightness_factor).plot.imshow(rgb='wl')#, subplot_kws=dict(projection= l1c.proj))
coarsening=1

gamma=0.2
brightness_factor = 1
#fig = (process_.l1c_prod.Rtoa[:, ::coarsening, ::coarsening].isel(wl=[30,20,10])**gamma*brightness_factor).plot.imshow(rgb='wl',robust=True)#, subplot_kws=dict(projection= l1c.proj))
fig =(process_.l2_prod.Rrs[:, ::coarsening, ::coarsening].sel(wl=[670,550,490],method='nearest')).plot.imshow(rgb='wl',robust=True)#, subplot_kws=dict(projection= l1c.proj))ax=fig.axes,
fig.axes.set(xticks=[], yticks=[])
fig.axes.set_ylabel('')
fig.axes.set_xlabel('')
fig

In [ ]:
subset = process_.l2_prod.Rrs.sel(wl=[405,440,490,510,550,600,640,660,695,705,710,750],method='nearest')

fig = subset.plot.imshow(col='wl',col_wrap=4,vmin=0,robust=True,cmap=plt.cm.Spectral_r)
for ax in fig.axs.flat:
    ax.set(xticks=[], yticks=[])
    ax.set_ylabel('')
    ax.set_xlabel('')
fig

In [ ]:


param = 'Rrs'
raster = process_.l2_prod[param] 


#param = 'rho'
#raster = dc_l2c[param] 
cmap='Spectral_r'
#cmap='RdBu_r'
third_dim = 'wl'

wl= raster.wl.data
Nwl = len(wl)
ds = hv.Dataset(raster.persist())
im= ds.to(hv.Image, ['x', 'y'], dynamic=True).opts(cmap= cmap,colorbar=True,clim=(0,0.03)).hist(bin_range=(0,0.02)) 

polys = hv.Polygons([])
box_stream = hv.streams.BoxEdit(source=polys)
dmap, dmap_std=[],[]

def roi_curves(data,ds=ds):    
    if not data or not any(len(d) for d in data.values()):
        return hv.NdOverlay({0: hv.Curve([],'Wavelength (nm)', param)})

    curves,envelope = {},{}
    data = zip(data['x0'], data['x1'], data['y0'], data['y1'])
    for i, (x0, x1, y0, y1) in enumerate(data):
        selection = ds.select(x=(x0, x1), y=(y0, y1))
        mean = selection.aggregate(third_dim, np.mean).data
        std = selection.aggregate(third_dim, np.std).data
        wl = mean.wl

        curves[i]= hv.Curve((wl,mean[param]),'Wavelength (nm)', param) 

    return hv.NdOverlay(curves)


# a bit dirty to have two similar function, but holoviews does not like mixing Curve and Spread for the same stream
def roi_spreads(data,ds=ds):    
    if not data or not any(len(d) for d in data.values()):
        return hv.NdOverlay({0: hv.Curve([],'Wavelength (nm)', param)})

    curves,envelope = {},{}
    data = zip(data['x0'], data['x1'], data['y0'], data['y1'])
    for i, (x0, x1, y0, y1) in enumerate(data):
        selection = ds.select(x=(x0, x1), y=(y0, y1))
        mean = selection.aggregate(third_dim, np.mean).data
        std = selection.aggregate(third_dim, np.std).data
        wl = mean.wl

        curves[i]=  hv.Spread((wl,mean[param],std[param]))
        #curves[i].opts(fill_alpha=0.3)
    return hv.NdOverlay(curves)

mean=hv.DynamicMap(roi_curves,streams=[box_stream])
std =hv.DynamicMap(roi_spreads, streams=[box_stream])    
hlines = hv.HoloMap({wl[i]: hv.VLine(wl[i]) for i in range(Nwl)},third_dim )


hv.output(widget_location='top_left')

# visualize and play
graphs = ((mean* std *hlines).relabel(param))
layout = (im * polys +graphs    ).opts(
    opts.Curve(width=750,height=500, framewise=True,xlim=(400,1100)), 
    opts.Polygons(fill_alpha=0.2, color='green',line_color='black'), 
    opts.VLine(color='black')).cols(2)
layout 